<a href="https://colab.research.google.com/github/Atif-04/atifkhaleel_augustbatch/blob/main/Rag_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install google-genai pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 7.7 MB/s eta 0:00:00


In [ ]:
from google import genai
from google.genai import types
from google.colab import userdata
import numpy as np

client = genai.Client(api_key=userdata.get('GEMINI_API_KEYS'))

In [ ]:
from google.colab import files
uploaded = files.upload()

pdf_name = list(uploaded.keys())[0]
print("is uploadedd?",pdf_name)

Saving college_cafeteria_rules.pdf to college_cafeteria_rules.pdf
is uploadedd? college_cafeteria_rules.pdf


In [ ]:
from pypdf import PdfReader
reader = PdfReader(pdf_name)
print("no of pages:",len(reader.pages))
text=""
for page in reader.pages:
  text+=page.extract_text() + "\n"
print("no of characters:",len(text))
print("first few characters:",text[:20])

no of pages: 2
no of characters: 2512
first few characters: Greenfield College C


In [ ]:
#chunking
def chunk_text(text,chunk_size=800,overlap=150):
  chunks = []
  start = 0
  while start<len(text):
    end=start+chunk_size
    chunks.append(text[start:end])
    start=end-overlap
  return chunks
chunks= chunk_text(text)
print("no of chunks:",len(chunks))
print("first chunk:",chunks[1])


no of chunks: 4
first chunk:  30.
4. Lunch
Lunch is served from 12:30 PM to 3:00 PM. A standard lunch includes rice or two chapatis, one vegetable
dish, dal, curd, and a salad. Students may request a vegetarian meal without additional charge.
5. Special Meals
A special meal is available every Wednesday for Rs. 120. The Wednesday special menu is announced on the
cafeteria notice board by Monday afternoon.
6. Payment Rules
The cafeteria accepts cash, UPI, and college meal cards. Credit purchases are not allowed. Students must
keep their payment receipt until they leave the cafeteria.
7. Cleanliness Rules
Students must return used plates, cups, and cutlery to the collection counter. Food waste must be placed in
the marked bins. Throwing food or waste on tables or floors is prohibited.
8. Seating Rules
A student may occup


In [ ]:
def embed_chunks(t):
  response = client.models.embed_content(
      model='gemini-embedding-001',
      contents=t,
      config = types.EmbedContentConfig(
          output_dimensionality=768
      ))

  return np.array(response.embeddings[0].values)

chunk_embeddings = []
for i,ch in enumerate(chunks):
    chunk_embeddings.append(embed_chunks(ch))
    print(f"embedding chunks:{i+1}/{len(chunks)}")
chunk_embeddings = np.array(chunk_embeddings)
print("chunk embedding shape:",chunk_embeddings.shape)



embedding chunks:1/4
embedding chunks:2/4
embedding chunks:3/4
embedding chunks:4/4
chunk embedding shape: (4, 768)


In [ ]:
chunk_embeddings = chunk_embeddings/np.linalg.norm(chunk_embeddings,axis=1,keepdims=True)
print("store",len(chunks),"chunks in vector store",chunk_embeddings.shape)

store 4 chunks in vector store (4, 768)


In [ ]:
def retrieve(question,k=2):
  question_embedding = embed_chunks(question)
  question_embedding = question_embedding/np.linalg.norm(question_embedding)
  scores = chunk_embeddings @ question_embedding
  top_index = np.argsort(scores)[::-1][:k]
  return [(chunks[i],scores[i]) for i in top_index]

In [ ]:
results = retrieve("How much does a lost meal card replacement cost?")
for chunk,score in results:
  print(f"score:{score:.3f}")
  print(chunk)

score:0.618
 reported to the cafeteria supervisor on the same day. Refund requests for
incorrect or missing items must be submitted within 24 hours of purchase.

11. Contact
The cafeteria supervisor is available at the main counter from 10:00 AM to 4:00 PM on working days.
Students can submit written suggestions in the feedback box near the entrance.
Quick Reference
 Item
Information
Monthly meal card
Rs. 2,500
Lost card replacement
Rs. 100
Wednesday special meal
Rs. 120
Extra breakfast serving
Rs. 30
Peak lunch table time
30 minutes
Refund request deadline
24 hours


score:0.589
Greenfield College Cafeteria Rules & Information
1. Cafeteria Timings
The college cafeteria is open from 8:00 AM to 6:00 PM from Monday to Friday. On Saturdays, it operates
from 9:00 AM to 2:00 PM. The cafeteria is closed on Sundays and public holidays.
2. Student Meal Card
Students can purchase a monthly meal card for Rs. 2,500. The meal card is valid for one calendar month and
cannot be transferred to anothe

In [ ]:

def ask(question,k=2):
  top = retrieve(question,k)

  contents = "\n\n".join(chunk for chunk,score in top)
  prompt = f"""take the context from the document
  {contents}
  question:{question}
  Answer only using the context above"""

  response = client.models.generate_content(
      model = 'gemini-3.1-flash-lite',
      contents =prompt,
      config = types.GenerateContentConfig(
    system_instruction = (
        "you are a helpul ai assistant that helps find information from the documet"
        "if the answer is not in the document reply :i dont know "

    ),
    temperature = 0.4
    )
  )

  return response.text

In [ ]:
user_input = input("enter your question:")
answer = ask(user_input)
print(answer)

enter your question:lunch time
Lunch is served from 12:30 PM to 3:00 PM.
